<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-02-python-for-document-processing/notebooks/Ophelie_JSON_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JSON Task — Working with Nested Order Data

Loads nested customer/order JSON, extracts order totals, flattens it with Pandas, and saves the result as a new JSON file.

**Setup:** this cell writes `sample.json` into the Colab session so the notebook is self-contained and re-runnable.

In [ ]:
%%writefile sample.json
[
  {
    "customer_id": 101,
    "name": "Alice Johnson",
    "email": "alice@example.com",
    "orders": [
      {
        "order_id": 5001,
        "date": "2024-02-01",
        "total": 120.50,
        "items": [
          {"product": "Laptop", "price": 1000.00, "quantity": 1},
          {"product": "Mouse", "price": 20.50, "quantity": 1}
        ],
        "payment": {
          "method": "Credit Card",
          "status": "Completed"
        }
      }
    ]
  },
  {
    "customer_id": 102,
    "name": "Bob Williams",
    "email": "bob@example.com",
    "orders": [
      {
        "order_id": 5002,
        "date": "2024-02-03",
        "total": 250.00,
        "items": [
          {"product": "Smartphone", "price": 700.00, "quantity": 1},
          {"product": "Charger", "price": 50.00, "quantity": 2}
        ],
        "payment": {
          "method": "PayPal",
          "status": "Pending"
        }
      },
      {
        "order_id": 5003,
        "date": "2024-02-07",
        "total": 75.00,
        "items": [
          {"product": "Headphones", "price": 75.00, "quantity": 1}
        ],
        "payment": {
          "method": "Credit Card",
          "status": "Completed"
        }
      }
    ]
  }
]


Writing sample.json


## Task 1 — Load the JSON file with the `json` library

In [ ]:
import json

with open('sample.json', 'r') as f:
    data = json.load(f)

data

[{'customer_id': 101,
  'name': 'Alice Johnson',
  'email': 'alice@example.com',
  'orders': [{'order_id': 5001,
    'date': '2024-02-01',
    'total': 120.5,
    'items': [{'product': 'Laptop', 'price': 1000.0, 'quantity': 1},
     {'product': 'Mouse', 'price': 20.5, 'quantity': 1}],
    'payment': {'method': 'Credit Card', 'status': 'Completed'}}]},
 {'customer_id': 102,
  'name': 'Bob Williams',
  'email': 'bob@example.com',
  'orders': [{'order_id': 5002,
    'date': '2024-02-03',
    'total': 250.0,
    'items': [{'product': 'Smartphone', 'price': 700.0, 'quantity': 1},
     {'product': 'Charger', 'price': 50.0, 'quantity': 2}],
    'payment': {'method': 'PayPal', 'status': 'Pending'}},
   {'order_id': 5003,
    'date': '2024-02-07',
    'total': 75.0,
    'items': [{'product': 'Headphones', 'price': 75.0, 'quantity': 1}],
    'payment': {'method': 'Credit Card', 'status': 'Completed'}}]}]

## Task 2 — All order IDs with their total amounts

In [ ]:
for customer in data:
    for order in customer['orders']:
        print(f"Order ID: {order['order_id']}, Total: {order['total']}")

Order ID: 5001, Total: 120.5
Order ID: 5002, Total: 250.0
Order ID: 5003, Total: 75.0


## Task 3 — Flatten with `pd.json_normalize`

`record_path=['orders', 'items']` makes one row per item (nested two levels deep); `meta` pulls the parent-level fields down onto each row.

In [ ]:
import pandas as pd

df = pd.json_normalize(
    data,
    record_path=['orders', 'items'],
    meta=['customer_id', ['orders', 'order_id'], ['orders', 'total']]
)
df = df.rename(columns={'orders.order_id': 'order_id', 'orders.total': 'total'})
df = df[['customer_id', 'order_id', 'product', 'quantity', 'total']]
df

,customer_id,order_id,product,quantity,total
0,101,5001,Laptop,1,120.5
1,101,5001,Mouse,1,120.5
2,102,5002,Smartphone,1,250.0
3,102,5002,Charger,2,250.0
4,102,5003,Headphones,1,75.0


## Task 4 — Save the processed DataFrame to a new JSON file

In [ ]:
df.to_json('orders_flat.json', orient='records', indent=2)
print(open('orders_flat.json').read())

[
  {
    "customer_id":101,
    "order_id":5001,
    "product":"Laptop",
    "quantity":1,
    "total":120.5
  },
  {
    "customer_id":101,
    "order_id":5001,
    "product":"Mouse",
    "quantity":1,
    "total":120.5
  },
  {
    "customer_id":102,
    "order_id":5002,
    "product":"Smartphone",
    "quantity":1,
    "total":250.0
  },
  {
    "customer_id":102,
    "order_id":5002,
    "product":"Charger",
    "quantity":2,
    "total":250.0
  },
  {
    "customer_id":102,
    "order_id":5003,
    "product":"Headphones",
    "quantity":1,
    "total":75.0
  }
]
